# Exploring the Data
BlueTide’s Customer Success team noticed some larger clients are reducing activity or leaving. The goal of this analysis is to identify patterns that signal churn in prepartion for Tableau:
- Product usage (commits, pipelines, active users)
- Subscription & revenue behavior
- Support ticket activity

We defined churn as:
- Status = 'Churned'
- Reduced usage/licensing
- Missed payments
- Increased or no support engagement

## Churn Overview
~20% churn rate in the dataset which is enough population to identify behavioral patterns.



In [ ]:
SELECT COUNT(*) AS total_companies
     , COUNT(CASE WHEN status = 'Churned' THEN 1 END) AS churned_companies
     , ROUND(COUNT(CASE WHEN status = 'Churned' THEN 1 END) * 100.0 / COUNT(*),2) AS churn_percent
FROM companies;

### Revenue Impact
Do churned companies represent high or low-value companies?

In [ ]:
CREATE OR REPLACE VIEW vw_revenue_summary AS
SELECT c.status
     , SUM(s.monthly_revenue) AS total_revenue
     , AVG(s.price_per_user * s.user_count) AS avg_monthly_revenue
FROM companies c
JOIN subscriptions s
ON c.company_id = s.company_id
GROUP BY c.status;

In [ ]:
SELECT * FROM vw_revenue_summary

Churned companies represent 21.37% of total revenue which suggests churn is concentrated in lower-value companies OR high-value companies leaving could be risky.

### Churn by Plan Type
That leads us to question: Do certain plans experience higher churn?

In [ ]:
CREATE OR REPLACE VIEW vw_churn_by_plan_heatmap AS
SELECT plan_type
     , status
     , COUNT(*) AS company_count
     , ROUND(COUNT(CASE WHEN status = 'Churned' THEN 1 END) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY plan_type), 2) AS churn_percent
FROM companies
GROUP BY plan_type, status
ORDER BY plan_type, status;

In [ ]:
SELECT * FROM vw_churn_by_plan_heatmap

This shows that the Free plan has the highest churn rate at 25.5%, while the Team plan has the lowest at 15.83%. This pattern aligns with typical customer behavior: Team plans are often adopted by companies more committed to the software, whereas Free plans tend to attract users who are testing the product or may be less engaged.

### Product Usage Comparison
Do churned companies use the platform less?

In [ ]:
CREATE OR REPLACE VIEW vw_product_usage_summary AS
SELECT c.status
     , COUNT(*) AS company_count
     , AVG(p.active_users) AS avg_active_users
     , AVG(p.commits) AS avg_commits
     , AVG(p.pipelines_run) AS avg_pipelines
     , AVG(p.deployments) AS avg_deployments
     , AVG(p.commits / NULLIF(p.active_users,0)) AS commits_per_user
FROM companies c
JOIN product_usage p
ON c.company_id = p.company_id
GROUP BY c.status;

In [ ]:
select * from vw_product_usage_summary

Churned companies show lower average active users, commits, pipelines and general declining usage which can indicate churn. 

### Product Usage Distribution (Histogram Prep)
Per-company usage metrics for distribution analysis in Tableau including active users, commits, pipelines, deployments, and commits per user.

In [ ]:
CREATE OR REPLACE VIEW vw_product_usage_distribution AS
SELECT c.status
     , p.active_users
     , p.commits
     , p.pipelines_run
     , p.deployments
     , (p.commits / NULLIF(p.active_users,0)) AS commits_per_user
FROM companies c
JOIN product_usage p
ON c.company_id = p.company_id;

In [ ]:
SELECT TOP 10 * 
FROM vw_product_usage_distribution

### Support Tickets
Do churned companies engage on average more/less with support?

In [ ]:
CREATE OR REPLACE VIEW vw_support_summary AS
SELECT c.status
     , COUNT(t.ticket_id) AS total_tickets
     , COUNT(t.ticket_id) / COUNT(DISTINCT c.company_id) AS avg_tickets_per_company
FROM companies c
LEFT JOIN support_tickets t
ON c.company_id = t.company_id
GROUP BY c.status;

In [ ]:
SELECT * FROM vw_support_summary

Churned companies show slightly higher support ticket activity. A closer look at ticket topics could help identify friction points leading to churn.

### Payment Behavior
What payment tendencies do churned companies have?
- Late payments?
- Failed payments?
- No payments?
- No autorenewal?

In [ ]:
CREATE OR REPLACE VIEW vw_payment_behavior AS
SELECT c.status
     , COUNT(CASE WHEN s.payment_status = 'Late' THEN 1 END) AS late_payments
     , AVG(s.price_per_user * s.user_count) AS avg_monthly_revenue
     , SUM(CASE WHEN s.auto_renew = TRUE THEN 1 ELSE 0 END) AS auto_renew_count
FROM companies c
JOIN subscriptions s
ON c.company_id = s.company_id
GROUP BY c.status;

In [ ]:
SELECT * FROM vw_payment_behavior

Unexpectedly, churned companies have fewer late payments but contribute higher average monthly revenue. This may indicate that high-value customers leave despite good payment behavior, suggesting other churn drivers such as product engagement or support experience.

### Company Size and Churn
Are smaller companies more likely to churn?

In [ ]:
CREATE OR REPLACE VIEW vw_company_size AS
SELECT status
     , AVG(company_size) AS avg_company_size
     , MIN(company_size) AS min_company_size
     , MAX(company_size) AS max_company_size
FROM companies
GROUP BY status;

In [ ]:
SELECT * FROM vw_company_size

The average company size between active and churned customers is fairly similar. We mightly look at the outliers (companies with significantly high or lower user counts) to determine the significance.

From here, Snowflake will be conneted to Tableau to create the visualizations. 

